# Module 8: Single Agent

Build a Decision Intelligence agent with tools, run it, and inspect the agentic loop in action. By the end of this module you will have a working agent that can look up company data, market benchmarks, and competitor information: and you will understand exactly what happens inside the loop at each step.

**Prerequisites:** Python 3.10+, AWS credentials configured (Strands uses Amazon Bedrock by default)

> **This workshop uses Strands Agents**: the open-source agent harness SDK. The same patterns carry over to other agent frameworks.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns NovaCart's financial and operational data | Looks up by lowercased name; returns CLV, churn rate, revenue, top-spender segment |
| `get_market_benchmarks(industry)` | Returns industry benchmark data for e-commerce | Returns avg CLV, churn, subscription adoption rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns a competitor's premium tier details | Returns pricing, pilot approach, adoption rate, CLV lift, time to profitability |

All three tools use mock data: no external APIs or keys needed. Swap them for real data sources to ground the agent in live market intelligence.

In [ ]:
# Install dependencies (takes ~20 seconds on first run)
!uv pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Pick ONE option. Pass model=<your_model> to Agent(...) to activate it.
#
# Option 1 — Claude Sonnet 4 (default, best quality):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
#
# Option 2 — Claude Haiku 4.5 (faster, lower cost):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
#
# Option 3 — Amazon Nova Pro (AWS credits / sponsored events):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
#
# Option 4 — Amazon Nova Lite (cheapest, AWS credits):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
#
# Without model= , Strands uses Claude Sonnet 4 via Bedrock by default.

---

## Part 1: Define Tools

Tools are Python functions decorated with `@tool`. The LLM reads the **docstring** to decide when and how to call them: the docstring is the routing logic, not code.

**Rule:** Write docstrings for the model, not for other developers. Be explicit about what the tool returns and when to use it.

In [ ]:
from strands import Agent, tool

# ── Mock data ─────────────────────────────────────────────────────────────
# Simulates a business intelligence backend for NovaCart —
# a mid-size e-commerce company (2M users) evaluating a Premium Tier launch.

COMPANIES = {
    "novacart": {
        "company": "NovaCart",
        "industry": "e-commerce",
        "active_users": 2_000_000,
        "annual_churn_rate_pct": 22,
        "average_order_value_usd": 85.00,
        "customer_lifetime_value_usd": 340.00,
        "annual_revenue_usd": 142_000_000,
        "subscription_revenue_usd": 0,
        "top_spender_segment_pct": 10,
        "top_spender_avg_order_value_usd": 210.00,
        "q1_survey_premium_interest_pct": 38,
    }
}

BENCHMARKS = {
    "e-commerce": {
        "avg_annual_churn_rate_pct": 25,
        "avg_customer_lifetime_value_usd": 290.00,
        "subscription_adoption_rate_pct": 31,
        "avg_subscription_price_usd": 15.50,
        "premium_tier_adoption_top_spenders_pct": 68,
        "clv_lift_with_subscription_pct": "20-35",
        "avg_time_to_subscription_profitability_months": 9,
        "key_insight": (
            "E-commerce brands with subscription tiers see 20-35% higher CLV vs "
            "non-subscribers. Top-spender segments adopt at 2x the base rate."
        ),
    }
}

COMPETITORS = {
    "shopmart": {
        "tier_name": "ShopMart Plus",
        "launched": "Q2 2024",
        "monthly_price_usd": 16.99,
        "pilot_approach": "5% of users for 90 days before full rollout",
        "adoption_rate_6mo_pct": 28,
        "clv_lift_pct": 22,
        "time_to_profitability_months": 8,
        "key_insight": "5% pilot + validate unit economics before scaling → 28% adoption in 6 months.",
    },
    "primestore": {
        "tier_name": "PrimeStore Unlimited",
        "launched": "Q4 2023",
        "monthly_price_usd": 12.99,
        "pilot_approach": "Full launch to all users with 30-day free trial",
        "adoption_rate_6mo_pct": 19,
        "clv_lift_pct": 14,
        "time_to_profitability_months": 14,
        "key_insight": "Full launch + free trial → higher sign-ups but 40% churned post-trial. Slower ROI.",
    },
}


@tool
def get_company_data(company_name: str) -> str:
    '''Get current financial and operational data for a company.

    Args:
        company_name: The company name to look up (e.g. 'NovaCart')
    '''
    key = company_name.lower().strip()
    data = COMPANIES.get(key)
    if not data:
        return f"No data found for '{company_name}'. Available: {', '.join(COMPANIES)}"
    return (
        f"Company: {data['company']} ({data['industry']})"
        f"\nActive users: {data['active_users']:,}"
        f"\nAnnual churn rate: {data['annual_churn_rate_pct']}%"
        f"\nAverage order value: ${data['average_order_value_usd']:.2f}"
        f"\nCustomer Lifetime Value (CLV): ${data['customer_lifetime_value_usd']:.2f}"
        f"\nAnnual revenue: ${data['annual_revenue_usd']:,.0f}"
        f"\nSubscription revenue: ${data['subscription_revenue_usd']:,.0f}"
        f"\nTop {data['top_spender_segment_pct']}% spenders avg order: ${data['top_spender_avg_order_value_usd']:.2f}"
        f"\nQ1 survey — premium interest (top spenders): {data['q1_survey_premium_interest_pct']}%"
    )


@tool
def get_market_benchmarks(industry: str) -> str:
    '''Get industry benchmarks and performance data for a given market sector.

    Args:
        industry: The industry sector (e.g. 'e-commerce')
    '''
    key = industry.lower().strip()
    data = BENCHMARKS.get(key)
    if not data:
        return f"No benchmarks for '{industry}'. Available: {', '.join(BENCHMARKS)}"
    return (
        f"Industry benchmarks: {key}"
        f"\nAvg annual churn rate: {data['avg_annual_churn_rate_pct']}%"
        f"\nAvg CLV: ${data['avg_customer_lifetime_value_usd']:.2f}"
        f"\nSubscription adoption rate: {data['subscription_adoption_rate_pct']}%"
        f"\nAvg subscription price: ${data['avg_subscription_price_usd']:.2f}/mo"
        f"\nPremium tier adoption (top spenders): {data['premium_tier_adoption_top_spenders_pct']}%"
        f"\nCLV lift with subscription: +{data['clv_lift_with_subscription_pct']}%"
        f"\nAvg time to profitability: {data['avg_time_to_subscription_profitability_months']} months"
        f"\nKey insight: {data['key_insight']}"
    )


@tool
def get_competitor_data(competitor_name: str) -> str:
    '''Get information about a competitor's premium subscription tier.

    Args:
        competitor_name: The competitor name (e.g. 'shopmart', 'primestore')
    '''
    key = competitor_name.lower().strip()
    data = COMPETITORS.get(key)
    if not data:
        return f"No data for '{competitor_name}'. Available: {', '.join(COMPETITORS)}"
    return (
        f"Tier: {data['tier_name']} (launched {data['launched']})"
        f"\nPrice: ${data['monthly_price_usd']:.2f}/mo"
        f"\nPilot approach: {data['pilot_approach']}"
        f"\n6-month adoption rate: {data['adoption_rate_6mo_pct']}%"
        f"\nCLV lift observed: +{data['clv_lift_pct']}%"
        f"\nTime to profitability: {data['time_to_profitability_months']} months"
        f"\nKey insight: {data['key_insight']}"
    )

---

## Part 2: Create and Run the Agent

Wire the tools into an `Agent` with a system prompt. The LLM uses both the system prompt and the tool docstrings to decide what to do at each step.

In [ ]:
SYSTEM_PROMPT = '''You are a Decision Intelligence Analyst for a technology company.
You help business leaders gather data and context before making strategic decisions.
Use your available tools to look up company data, market benchmarks, and competitor information.

Guidelines:
- Always use tools to answer questions — never guess when real data is available.
- Be concise and data-driven.
- Surface the most relevant numbers for the decision at hand.'''

agent = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=SYSTEM_PROMPT,
)

# The agent decides which tools to call based on the request and tool docstrings.
result = agent(
    "What is NovaCart's current CLV and churn rate? "
    "How does that compare to the e-commerce industry benchmark?"
)

---

## Part 3: Inspect the Agent Loop

The agentic loop is:

```
User message → LLM reasons → selects a tool → tool executes → result appended to context → LLM reasons again → ... repeat until done
```

Every step is stored in `agent.messages`. Let's see what actually happened.

In [ ]:
import json as _json

print(f"Total messages in conversation: {len(agent.messages)}")
print("=" * 65)

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    content = msg.get("content", [])

    if role == "user":
        # content can be a string or a list of content blocks
        if isinstance(content, str):
            text = content[:100]
        elif content:
            text = str(content[0].get("text", content[0]))[:100]
        else:
            text = ""
        print(f"\n[{i}] USER:         {text}")

    elif role == "assistant":
        for block in content:
            if "text" in block:
                print(f"\n[{i}] ASSISTANT:    {block['text'][:120]}...")
            elif "toolUse" in block:
                tu = block["toolUse"]
                inp = _json.dumps(tu.get("input", {}))
                print(f"\n[{i}] TOOL CALL:    {tu['name']}({inp})")

    elif role == "tool":
        for block in content:
            result_text = str(block.get("content", ""))[:120]
            print(f"\n[{i}] TOOL RESULT:  {result_text}...")

print("\n" + "=" * 65)

In [ ]:
# AgentResult carries usage metrics for the last invocation
summary = result.metrics.get_summary()
print("Loop metrics:")
print(f"  Cycles (LLM calls):  {summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {summary.get('input_tokens', 'n/a')}")
print(f"  Output tokens:       {summary.get('output_tokens', 'n/a')}")

---

## Part 4: Try It Yourself

The agent keeps its conversation history across cells: each call to `agent(...)` adds to the same context window. Try the prompts below in order to see multi-turn behavior.

In [ ]:
# The agent keeps conversation history across cells — each call adds to the same context.
# Ask a follow-up using the NovaCart context already in memory:
agent("What did ShopMart do when they launched their premium tier? How did it perform?")

In [ ]:
agent("And PrimeStore? Which approach — ShopMart's or PrimeStore's — had better ROI?")

In [ ]:
# Uncomment to explore further:
# agent("What percentage of NovaCart's top spenders said they'd pay for a premium tier?")
# agent("Based on industry benchmarks, how long should NovaCart expect before subscription profit?")

In [ ]:
import logging

# ── Lens 1: Strands DEBUG logging ────────────────────────────────────────
# Enable this to see every reasoning step, tool call, and token event.
# Turn it off after to keep the next cells clean.

logging.getLogger("strands").setLevel(logging.DEBUG)
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
    force=True,
)

# Run a fresh agent so you can watch the loop steps in the output below
obs_agent = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=SYSTEM_PROMPT,
)
obs_result = obs_agent("What is NovaCart CLV and how does it compare to the industry?")

# Turn debug logging back off
logging.getLogger("strands").setLevel(logging.WARNING)

In [ ]:
# ── Lens 2: result.metrics.get_summary() ─────────────────────────────────
# Zero extra config — token usage, cycle count, and per-tool stats
# are on every AgentResult your agent call returns.
import json as _json

summary = obs_result.metrics.get_summary()

print("=== Loop metrics ===")
print(f"  total_cycles:     {summary.get('total_cycles')}")
print(f"  total_duration_s: {round(summary.get('total_duration', 0), 2)}")
print()

usage = summary.get("accumulated_usage", {})
print("=== Token usage ===")
print(f"  input_tokens:  {usage.get('inputTokens', 'n/a')}")
print(f"  output_tokens: {usage.get('outputTokens', 'n/a')}")
print(f"  total_tokens:  {usage.get('totalTokens', 'n/a')}")
print()

tool_usage = summary.get("tool_usage", {})
if tool_usage:
    print("=== Per-tool stats ===")
    for tool_name, data in tool_usage.items():
        stats = data.get("execution_stats", {})
        print(
            f"  {tool_name}: "
            f"calls={stats.get('call_count', 0)} | "
            f"success={stats.get('success_count', 0)} | "
            f"avg_time={round(stats.get('average_time', 0), 3)}s"
        )
else:
    print("(no tool_usage in summary — the agent answered from memory)")

---

## Part 5: The Ceiling: One Agent, One Giant Context

So far the agent handled focused questions well. Now let's give it the **full Decision Brief**: three options, competitive context, financial targets, timeline: and ask for a complete recommendation.

Watch what happens: the agent tries to do everything in a single context window.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Context:
- Current annual churn rate: 22%  |  Industry avg: 25%
- Customer Lifetime Value (CLV): $340  |  Industry avg: $290
- A major competitor launched a similar tier last quarter
- Q1 survey: 38% of top-spending users expressed interest in a premium tier

Options to evaluate:
  Option A — Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B — Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C — Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

For each option provide:
  - Pros and cons
  - Implementation complexity (Low / Medium / High)
  - Top 3 risks with mitigations
  - Revenue and CLV impact estimate

Then recommend the best option with justification.

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# Fresh agent with no prior context — clean baseline
ceiling_agent = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=SYSTEM_PROMPT,
)

result_ceiling = ceiling_agent(DECISION_BRIEF)

In [ ]:
# How many messages and tokens did the full brief take?
ceiling_summary = result_ceiling.metrics.get_summary()
print("Single-agent on full brief — metrics:")
print(f"  Cycles (LLM calls):  {ceiling_summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {ceiling_summary.get('input_tokens', 'n/a')}")
print(f"  Output tokens:       {ceiling_summary.get('output_tokens', 'n/a')}")
print(f"  Messages in context: {len(ceiling_agent.messages)}")
print()
print("One agent played: Researcher + Option Analyst x3 + Synthesizer.")
print("As complexity grows the context bloats and the model loses focus.")
print("That is the ceiling. Module 2 shows how to break past it.")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` | Python function the LLM can call: docstring is the routing logic |
| `Agent(tools=[], system_prompt=...)` | Assembles the harness: model + tools + instructions |
| `agent.messages` | Full conversation history: every user turn, LLM response, tool call, tool result |
| `result.metrics` | Loop execution stats: cycles and tokens consumed |
| **The ceiling** | One agent juggling researcher + analyst + synthesizer roles bloats the context |

---

## What's Next

In **Module 8: Sequential Chain**, you break this into a clean pipeline: each stage passes its output to the next as a focused handoff. Same task, three specialized agents, predictable and debuggable flow.

---

## Want a real multi-turn conversation?

Each notebook cell is a single turn. To chat back and forth with the agent maintaining memory across turns, run the companion script in a terminal:

```bash
cd samples/02-single-agent
uv pip install -r requirements.txt
uv run python chat.py
```

Type your messages, `quit` or Ctrl+C to exit.